# Task 2 — Unified 5-Seed Evaluation: Standard A2C vs. A2C_mod vs. DQN vs. Base-Stock

- **Mục tiêu:** Đánh giá công bằng (apples-to-apples) trên cùng 1 môi trường 504 bước kiểm định × 4 kịch bản (`ALL`, `EASY`, `MEDIUM`, `HARD`) + trung bình 3 kịch bản `Mean(E/M/H)` cho:
  - **Base-Stock ($k = 1.0$)** (val-selected safety factor)
  - **DQN** (5 seed: `42, 123, 256, 512, 1024`)
  - **`A2C_mod`** (5 seed: `42, 123, 256, 512, 1024`)
  - **Standard A2C** (5 seed: `42, 123, 256, 512, 1024`, `ckpt-60` mỗi seed)
- **Đường dẫn checkpoint:** Tường minh bằng **đường dẫn tuyệt đối** cho toàn bộ 15 checkpoint.
- **Đầu ra (`Feedback 23-11/task2/output/`):**
  - `task2_5seed_all_models_summary.csv` (cho **Table 6**)
  - `task2_table7_deltas.csv` ($\Delta$ so với Base-Stock cho **Table 7**)
  - `task2_a2c_vs_a2cmod_paired_tests.csv` (kiểm định ghép cặp 5 seed Standard A2C vs. A2C_mod)
  - `task2_policy_diagnostics.csv` (entropy, max prob, tỷ lệ action cực trị / policy collapse cho từng seed)
  - `task2_summary.json`


In [1]:
import os, json, pathlib, warnings
import numpy as np
import pandas as pd
from scipy import stats
import tensorflow as tf

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ==============================================================================
# 1. EXPLICIT ABSOLUTE PATHS (Data & All 15 Checkpoints)
# ==============================================================================
REPO_ROOT = pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management")
DATA_DIR  = REPO_ROOT / "data"
TASK2_DIR = REPO_ROOT / "Feedback 23-11" / "task2"
OUT_DIR   = TASK2_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE    = str(DATA_DIR / "train.tfrecords")
TEST_FILE     = str(DATA_DIR / "test.tfrecords")
CAPACITY_FILE = str(DATA_DIR / "capacity.tfrecords")
STOCK_FILE    = str(DATA_DIR / "stock.tfrecords")

SEEDS_5 = [42, 123, 256, 512, 1024]

STD_A2C_CKPT_DIRS = {
    42:   pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task2\A2C_42\checkpoints"),
    123:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task2\A2C_123\checkpoints"),
    256:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task2\A2C_256\checkpoints"),
    512:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task2\A2C_512\checkpoints"),
    1024: pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task2\A2C_1024\checkpoints"),
}

A2C_MOD_CKPTS = {
    42:   pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\checkpoints_a2c_42\ckpt-53"),
    123:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\A2C_mod_123\checkpoints\ckpt-60"),
    256:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\A2C_mod_256\checkpoints\ckpt-60"),
    512:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\checkpoints_a2c_512\ckpt-60"),
    1024: pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\checkpoints_a2c_1024\ckpt-60"),
}

DQN_CKPT_DIRS = {
    42:   pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\dqn\checkpoints_dqn_comparison42"),
    123:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\dqn\checkpoints_dqn_comparison123"),
    256:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\dqn\checkpoints_dqn_comparison256"),
    512:  pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\dqn\checkpoints_dqn_comparison512"),
    1024: pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management\dqn\checkpoints_dqn_comparison1024"),
}

NUM_PRODUCTS   = 220
NUM_ACTIONS    = 14
ZERO_INVENTORY = 1e-5
ACTION_LEVELS  = np.array(
    [0.0, 0.005, 0.01, 0.0125, 0.015, 0.0175, 0.02, 0.03, 0.04, 0.08, 0.12, 0.2, 0.5, 1.0],
    dtype=np.float32
)

SCENARIOS = {
    "ALL":    {"x_scale": 1.0, "sales_scale": 1.0, "waste_rate": 0.025},
    "EASY":   {"x_scale": 0.3, "sales_scale": 0.2, "waste_rate": 0.010},
    "MEDIUM": {"x_scale": 0.6, "sales_scale": 0.5, "waste_rate": 0.050},
    "HARD":   {"x_scale": 0.9, "sales_scale": 0.8, "waste_rate": 0.150},
}

METRICS = ["total_reward", "service_level", "holding_cost", "waste_cost", "ordering_cost", "stockout_rate"]

print("TensorFlow:", tf.__version__)
print("Output dir:", OUT_DIR)


TensorFlow: 2.20.0
Output dir: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task2\output


In [2]:
# ==============================================================================
# 2. LOAD TFRECORD DATA (Train, Test, Capacity, Initial Stock)
# ==============================================================================
def _parse_vec(serialized, key):
    ex = tf.io.parse_single_example(serialized, {key: tf.io.FixedLenFeature([NUM_PRODUCTS], tf.float32)})
    return ex[key]

capacity  = next(iter(tf.data.TFRecordDataset(CAPACITY_FILE).map(lambda s: _parse_vec(s, "capacity")))).numpy()
x_init    = next(iter(tf.data.TFRecordDataset(STOCK_FILE).map(lambda s: _parse_vec(s, "stock")))).numpy()
train_raw = np.array([r.numpy() for r in tf.data.TFRecordDataset(TRAIN_FILE).map(lambda s: _parse_vec(s, "sales"))], dtype=np.float32)
test_raw  = np.array([r.numpy() for r in tf.data.TFRecordDataset(TEST_FILE).map(lambda s: _parse_vec(s, "sales"))], dtype=np.float32)

train_sales = train_raw / capacity[None, :]
test_sales  = test_raw  / capacity[None, :]

# Base-Stock target (k = 1.0, fitted on training demand over capacity)
bs_mean   = np.mean(train_sales, axis=0)
bs_std    = np.std(train_sales, axis=0)
bs_target = np.clip(bs_mean + 1.0 * bs_std, 0.0, 1.0).astype(np.float32)

print(f"Train sales: {train_sales.shape} | Test sales: {test_sales.shape}")
print(f"Capacity mean: {capacity.mean():.4f} | x_init mean: {x_init.mean():.4f} | Base-Stock target mean: {bs_target.mean():.4f}")


Train sales: (1000, 220) | Test sales: (504, 220)
Capacity mean: 20.3273 | x_init mean: 0.5049 | Base-Stock target mean: 0.2368


In [3]:
# ==============================================================================
# 3. MODEL DEFINITIONS & DIRECT CHECKPOINT LOADERS (Keras 3 / TF 2.20 Safe)
# ==============================================================================
class ActorNet(tf.keras.Model):
    def __init__(self, hidden_size=32, num_actions=14):
        super().__init__()
        self.layer1 = tf.keras.layers.Dense(hidden_size, activation="relu")
        self.layer2 = tf.keras.layers.Dense(hidden_size, activation="relu")
        self.layer3 = tf.keras.layers.Dense(hidden_size, activation="relu")
        self.output_layer = tf.keras.layers.Dense(num_actions, activation="softmax")

    def call(self, x, training=False):
        h = self.layer1(x)
        h = self.layer2(h)
        h = self.layer3(h)
        return self.output_layer(h)

class MultiProductQNetwork(tf.keras.Model):
    def __init__(self, num_features=660, num_products=220, num_actions=14, hidden_size=128, dropout_prob=0.1, use_group_norm=True):
        super().__init__()
        self.num_products = num_products
        self.num_actions = num_actions
        self.features_per_prod = num_features // num_products
        self.dense1 = tf.keras.layers.Dense(hidden_size, activation=None)
        self.dense2 = tf.keras.layers.Dense(hidden_size, activation=None)
        self.dense3 = tf.keras.layers.Dense(hidden_size, activation=None)
        self.out    = tf.keras.layers.Dense(num_actions, activation=None)
        self._use_gn = use_group_norm
        if use_group_norm:
            self.gn1 = tf.keras.layers.GroupNormalization(groups=1)
            self.gn2 = tf.keras.layers.GroupNormalization(groups=1)
            self.gn3 = tf.keras.layers.GroupNormalization(groups=1)
        self.drop1 = tf.keras.layers.Dropout(dropout_prob)
        self.drop2 = tf.keras.layers.Dropout(dropout_prob)
        self.drop3 = tf.keras.layers.Dropout(dropout_prob)

    def call(self, state, training=False):
        B = tf.shape(state)[0]
        P, F = self.num_products, self.features_per_prod
        s3d = tf.transpose(tf.reshape(state, [B, F, P]), [0, 2, 1])
        x = tf.reshape(s3d, [B * P, F])
        x = self.dense1(x)
        if self._use_gn: x = self.gn1(x, training=training)
        x = self.drop1(tf.nn.relu(x), training=training)
        x = self.dense2(x)
        if self._use_gn: x = self.gn2(x, training=training)
        x = self.drop2(tf.nn.relu(x), training=training)
        x = self.dense3(x)
        if self._use_gn: x = self.gn3(x, training=training)
        x = self.drop3(tf.nn.relu(x), training=training)
        return tf.reshape(self.out(x), [B, P, self.num_actions])

def load_actor_checkpoint(ckpt_path_or_dir, label=""):
    p = pathlib.Path(ckpt_path_or_dir)
    latest = tf.train.latest_checkpoint(str(p)) if p.is_dir() else str(p)
    assert latest is not None, f"No checkpoint in {p}"
    avail = {k: sh for k, sh in tf.train.list_variables(latest)}

    # Detect prefix & naming convention (layer1/w vs layer1/kernel)
    prefix = "actor/" if any(k.startswith("actor/layer1/") for k in avail) else ""
    w_name = "w" if f"{prefix}layer1/w/.ATTRIBUTES/VARIABLE_VALUE" in avail else "kernel"
    b_name = "b" if f"{prefix}layer1/b/.ATTRIBUTES/VARIABLE_VALUE" in avail else "bias"
    hidden_size = int(avail[f"{prefix}layer1/{w_name}/.ATTRIBUTES/VARIABLE_VALUE"][1])

    actor = ActorNet(hidden_size=hidden_size, num_actions=NUM_ACTIONS)
    _ = actor(tf.zeros((1, 3), dtype=tf.float32), training=False)
    w_before = float(tf.norm(actor.layer1.kernel))

    mapping = [
        (actor.layer1.kernel,       f"{prefix}layer1/{w_name}"),
        (actor.layer1.bias,         f"{prefix}layer1/{b_name}"),
        (actor.layer2.kernel,       f"{prefix}layer2/{w_name}"),
        (actor.layer2.bias,         f"{prefix}layer2/{b_name}"),
        (actor.layer3.kernel,       f"{prefix}layer3/{w_name}"),
        (actor.layer3.bias,         f"{prefix}layer3/{b_name}"),
        (actor.output_layer.kernel, f"{prefix}layer4/{w_name}"),
        (actor.output_layer.bias,   f"{prefix}layer4/{b_name}"),
    ]
    for var, key in mapping:
        full_key = f"{key}/.ATTRIBUTES/VARIABLE_VALUE"
        assert full_key in avail, f"Missing {full_key} in {latest}"
        var.assign(tf.train.load_variable(latest, full_key))

    w_after = float(tf.norm(actor.layer1.kernel))
    assert abs(w_after - w_before) > 1e-5, f"Actor weights unchanged for {latest}!"
    print(f"OK: {label:22s} restored -> {pathlib.Path(latest).name} (hidden={hidden_size}, layer1 norm={w_after:.4f})")
    return actor, pathlib.Path(latest).name

def load_dqn_checkpoint(ckpt_dir, label=""):
    ckpt_dir = pathlib.Path(ckpt_dir)
    latest = tf.train.latest_checkpoint(str(ckpt_dir))
    assert latest is not None, f"No checkpoint in {ckpt_dir}"
    avail_shapes = dict(tf.train.list_variables(latest))
    avail = set(avail_shapes.keys())
    prefix = "q_network" if any(n.startswith("q_network/") for n in avail) else "target_network"
    k_name = "kernel" if f"{prefix}/dense1/kernel/.ATTRIBUTES/VARIABLE_VALUE" in avail else "_kernel"
    hidden_size = int(avail_shapes[f"{prefix}/dense1/{k_name}/.ATTRIBUTES/VARIABLE_VALUE"][1])

    net = MultiProductQNetwork(hidden_size=hidden_size)
    _ = net(tf.zeros([1, 660], dtype=tf.float32), training=False)
    w_before = float(tf.norm(net.dense1.kernel))

    mapping = [
        (net.dense1.kernel, f"{prefix}/dense1/{k_name}"), (net.dense1.bias, f"{prefix}/dense1/bias"),
        (net.dense2.kernel, f"{prefix}/dense2/{k_name}"), (net.dense2.bias, f"{prefix}/dense2/bias"),
        (net.dense3.kernel, f"{prefix}/dense3/{k_name}"), (net.dense3.bias, f"{prefix}/dense3/bias"),
        (net.gn1.gamma,     f"{prefix}/gn1/gamma"),       (net.gn1.beta,    f"{prefix}/gn1/beta"),
        (net.gn2.gamma,     f"{prefix}/gn2/gamma"),       (net.gn2.beta,    f"{prefix}/gn2/beta"),
        (net.gn3.gamma,     f"{prefix}/gn3/gamma"),       (net.gn3.beta,    f"{prefix}/gn3/beta"),
        (net.out.kernel,    f"{prefix}/out/{k_name}"),    (net.out.bias,    f"{prefix}/out/bias"),
    ]
    for var, key in mapping:
        full_key = f"{key}/.ATTRIBUTES/VARIABLE_VALUE"
        assert full_key in avail, f"Missing {full_key} in {latest}"
        var.assign(tf.train.load_variable(latest, full_key))

    w_after = float(tf.norm(net.dense1.kernel))
    assert abs(w_after - w_before) > 1e-5, f"DQN weights unchanged for {latest}!"
    print(f"OK: {label:22s} restored -> {pathlib.Path(latest).name} (hidden={hidden_size}, dense1 norm={w_after:.4f})")
    return net, f"{pathlib.Path(latest).name} (hidden={hidden_size})"

std_a2c_models, std_a2c_info = {}, {}
a2c_mod_models, a2c_mod_info = {}, {}
dqn_models,     dqn_info     = {}, {}

for s in SEEDS_5:
    std_a2c_models[s], std_a2c_info[s] = load_actor_checkpoint(STD_A2C_CKPT_DIRS[s], f"Std-A2C [seed {s}]")
for s in SEEDS_5:
    a2c_mod_models[s], a2c_mod_info[s] = load_actor_checkpoint(A2C_MOD_CKPTS[s],     f"A2C_mod [seed {s}]")
for s in SEEDS_5:
    dqn_models[s], dqn_info[s]         = load_dqn_checkpoint(DQN_CKPT_DIRS[s],       f"DQN     [seed {s}]")


OK: Std-A2C [seed 42]      restored -> ckpt-60 (hidden=32, layer1 norm=7.9360)
OK: Std-A2C [seed 123]     restored -> ckpt-60 (hidden=32, layer1 norm=7.4436)
OK: Std-A2C [seed 256]     restored -> ckpt-60 (hidden=32, layer1 norm=7.6122)
OK: Std-A2C [seed 512]     restored -> ckpt-60 (hidden=32, layer1 norm=6.4339)
OK: Std-A2C [seed 1024]    restored -> ckpt-60 (hidden=32, layer1 norm=8.6072)
OK: A2C_mod [seed 42]      restored -> ckpt-53 (hidden=32, layer1 norm=8.3177)
OK: A2C_mod [seed 123]     restored -> ckpt-60 (hidden=32, layer1 norm=8.6017)
OK: A2C_mod [seed 256]     restored -> ckpt-60 (hidden=32, layer1 norm=9.0679)
OK: A2C_mod [seed 512]     restored -> ckpt-60 (hidden=32, layer1 norm=7.6431)
OK: A2C_mod [seed 1024]    restored -> ckpt-60 (hidden=32, layer1 norm=9.9056)
OK: DQN     [seed 42]      restored -> ckpt-50 (hidden=128, dense1 norm=23.9807)
OK: DQN     [seed 123]     restored -> ckpt-50 (hidden=128, dense1 norm=25.9539)
OK: DQN     [seed 256]     restored -> ckpt-50 (

In [4]:
# ==============================================================================
# 4. UNIFIED ROLLOUT EVALUATION & POLICY DIAGNOSTICS (504 TEST STEPS)
# ==============================================================================
def rollout_policy(policy_type, model, sc_cfg):
    x = np.clip(x_init * sc_cfg["x_scale"], 0.0, 1.0).astype(np.float32)
    T = len(test_sales)
    step_metrics = {m: np.zeros(T, dtype=np.float64) for m in METRICS}

    entropies = []
    max_probs = []
    action_counts = np.zeros(NUM_ACTIONS, dtype=np.int64)

    for t in range(T):
        sales_t = (test_sales[t] * sc_cfg["sales_scale"]).astype(np.float32)
        q_t     = (x * sc_cfg["waste_rate"]).astype(np.float32)

        if policy_type == "basestock":
            needed = np.maximum(0.0, bs_target - x)
            actions = np.argmin(np.abs(ACTION_LEVELS[None, :] - needed[:, None]), axis=1)
        elif policy_type == "dqn":
            state_flat = np.concatenate([x, sales_t, q_t], axis=0).astype(np.float32)[None, :]
            q_vals = model(tf.constant(state_flat, dtype=tf.float32), training=False).numpy()[0]
            actions = np.argmax(q_vals, axis=-1)
        elif policy_type == "actor":
            s_t = np.stack([x, sales_t, q_t], axis=1)
            probs = model(tf.constant(s_t, dtype=tf.float32), training=False).numpy()
            actions = np.argmax(probs, axis=-1)
            ent = -np.sum(probs * np.log(np.clip(probs, 1e-12, 1.0)), axis=-1)
            entropies.append(float(np.mean(ent)))
            max_probs.append(float(np.mean(np.max(probs, axis=-1))))
        else:
            raise ValueError(policy_type)

        np.add.at(action_counts, actions, 1)
        u = ACTION_LEVELS[actions]

        overstock = np.maximum(0.0, (x + u) - 1.0)
        x_u       = np.minimum(1.0, x + u)
        z         = (x < ZERO_INVENTORY).astype(np.float32)
        quan      = float(np.quantile(x, 0.95) - np.quantile(x, 0.05))
        r         = 1.0 - z - overstock - q_t - quan

        step_metrics["total_reward"][t]  = float(np.mean(r))
        step_metrics["service_level"][t] = float(np.mean(1.0 - z))
        step_metrics["holding_cost"][t]  = float(np.mean(overstock))
        step_metrics["waste_cost"][t]    = float(np.mean(q_t))
        step_metrics["ordering_cost"][t] = quan
        step_metrics["stockout_rate"][t] = float(np.mean(z))

        x = np.maximum(0.0, x_u - sales_t)

    summary = {m: float(np.mean(step_metrics[m])) for m in METRICS}
    dom_action = int(np.argmax(action_counts))
    dom_share  = float(action_counts[dom_action] / np.sum(action_counts))
    diag = {
        "mean_entropy":   float(np.mean(entropies)) if entropies else np.nan,
        "mean_max_prob":  float(np.mean(max_probs)) if max_probs else np.nan,
        "dom_action_idx": dom_action,
        "dom_action_pct": dom_share * 100.0,
        "act13_pct":      float(action_counts[13] / np.sum(action_counts)) * 100.0,
    }
    return summary, diag

# Run all evaluations
per_seed_rows = []
diag_rows     = []

for sc_name, sc_cfg in SCENARIOS.items():
    bs_sum, bs_diag = rollout_policy("basestock", None, sc_cfg)
    per_seed_rows.append({"model": "Base-Stock", "seed": 0, "scenario": sc_name, **bs_sum})

    for s in SEEDS_5:
        for mname, ptype, mdict in [
            ("DQN",          "dqn",   dqn_models),
            ("A2C_mod",      "actor", a2c_mod_models),
            ("Standard A2C", "actor", std_a2c_models),
        ]:
            msum, mdiag = rollout_policy(ptype, mdict[s], sc_cfg)
            per_seed_rows.append({"model": mname, "seed": s, "scenario": sc_name, **msum})
            if ptype == "actor":
                diag_rows.append({"model": mname, "seed": s, "scenario": sc_name, **mdiag})

df_per_seed = pd.DataFrame(per_seed_rows)

# Add 'Mean(E/M/H)' row per (model, seed) — matching Table 6 'All scenarios' definition!
emh_rows = []
for (mname, s), grp in df_per_seed[df_per_seed["scenario"].isin(["EASY", "MEDIUM", "HARD"])].groupby(["model", "seed"]):
    row = {"model": mname, "seed": s, "scenario": "Mean(E/M/H)"}
    for m in METRICS:
        row[m] = float(grp[m].mean())
    emh_rows.append(row)

df_per_seed = pd.concat([df_per_seed, pd.DataFrame(emh_rows)], ignore_index=True)
df_diag     = pd.DataFrame(diag_rows)

print("Done! Per-seed rows:", len(df_per_seed), "| Diagnostics rows:", len(df_diag))


Done! Per-seed rows: 80 | Diagnostics rows: 40


In [5]:
# ==============================================================================
# 5. AGGREGATE 5-SEED SUMMARY (TABLE 6), DELTAS VS BASE-STOCK (TABLE 7), & STATS
# ==============================================================================
SC_ORDER    = ["Mean(E/M/H)", "ALL", "EASY", "MEDIUM", "HARD"]
MODEL_ORDER = ["Base-Stock", "DQN", "A2C_mod", "Standard A2C"]

summary_rows = []
for sc in SC_ORDER:
    for mname in MODEL_ORDER:
        sub = df_per_seed[(df_per_seed["scenario"] == sc) & (df_per_seed["model"] == mname)]
        row = {"scenario": sc, "model": mname, "n_seeds": len(sub)}
        for m in METRICS:
            row[f"{m}_mean"] = float(sub[m].mean())
            row[f"{m}_sd"]   = float(sub[m].std(ddof=1)) if len(sub) > 1 else 0.0
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / "task2_5seed_all_models_summary.csv", index=False)

# Table 7 deltas vs Base-Stock (for both Mean(E/M/H) and ALL)
delta_rows = []
for sc in ["Mean(E/M/H)", "ALL"]:
    bs_row = df_summary[(df_summary["scenario"] == sc) & (df_summary["model"] == "Base-Stock")].iloc[0]
    for m in METRICS:
        entry = {"scenario": sc, "metric": m, "Base-Stock": bs_row[f"{m}_mean"]}
        for mname in ["DQN", "A2C_mod", "Standard A2C"]:
            r = df_summary[(df_summary["scenario"] == sc) & (df_summary["model"] == mname)].iloc[0]
            entry[f"{mname}_mean"]  = r[f"{m}_mean"]
            entry[f"{mname}_sd"]    = r[f"{m}_sd"]
            entry[f"{mname}_delta"] = r[f"{m}_mean"] - bs_row[f"{m}_mean"]
        delta_rows.append(entry)

df_deltas = pd.DataFrame(delta_rows)
df_deltas.to_csv(OUT_DIR / "task2_table7_deltas.csv", index=False)

# Paired 5-seed statistical tests: Standard A2C minus A2C_mod
def paired_stats_5(a, b):
    d = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    mean_d = float(np.mean(d))
    sd_d   = float(np.std(d, ddof=1))
    n      = len(d)
    t_p    = float(stats.ttest_rel(a, b).pvalue) if sd_d > 0 else 1.0
    try:
        w_p = float(stats.wilcoxon(d).pvalue) if np.any(d != 0) else 1.0
    except Exception:
        w_p = 1.0
    J = 1.0 - 3.0 / (4.0 * n - 1.0)
    hedges_g = float((mean_d / sd_d) * J) if sd_d > 0 else 0.0
    rng = np.random.default_rng(42)
    boots = np.mean(rng.choice(d, size=(5000, n), replace=True), axis=1)
    ci_lo, ci_hi = float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
    return mean_d, sd_d, ci_lo, ci_hi, t_p, w_p, hedges_g

test_rows = []
for sc in SC_ORDER:
    for m in METRICS:
        sub_std = df_per_seed[(df_per_seed["scenario"] == sc) & (df_per_seed["model"] == "Standard A2C")].sort_values("seed")[m].values
        sub_mod = df_per_seed[(df_per_seed["scenario"] == sc) & (df_per_seed["model"] == "A2C_mod")].sort_values("seed")[m].values
        md, sd, lo, hi, tp, wp, hg = paired_stats_5(sub_std, sub_mod)
        test_rows.append({
            "scenario": sc, "metric": m,
            "std_a2c_mean": float(np.mean(sub_std)), "std_a2c_sd": float(np.std(sub_std, ddof=1)),
            "a2c_mod_mean": float(np.mean(sub_mod)), "a2c_mod_sd": float(np.std(sub_mod, ddof=1)),
            "delta_std_minus_mod": md, "delta_sd": sd, "ci95_lo": lo, "ci95_hi": hi,
            "ttest_p": tp, "wilcoxon_p": wp, "hedges_g": hg,
        })

df_tests = pd.DataFrame(test_rows)
df_tests.to_csv(OUT_DIR / "task2_a2c_vs_a2cmod_paired_tests.csv", index=False)
df_diag.to_csv(OUT_DIR / "task2_policy_diagnostics.csv", index=False)

# Save full JSON summary
summary_json = {
    "std_a2c_ckpts": std_a2c_info,
    "a2c_mod_ckpts": a2c_mod_info,
    "dqn_ckpts": dqn_info,
    "per_seed_total_reward": df_per_seed[["model", "seed", "scenario", "total_reward", "holding_cost", "ordering_cost", "stockout_rate"]].to_dict(orient="records"),
    "summary_mean_emh": df_summary[df_summary["scenario"] == "Mean(E/M/H)"].to_dict(orient="records"),
    "summary_all": df_summary[df_summary["scenario"] == "ALL"].to_dict(orient="records"),
    "table7_deltas": df_deltas.to_dict(orient="records"),
    "paired_tests_mean_emh": df_tests[df_tests["scenario"] == "Mean(E/M/H)"].to_dict(orient="records"),
    "policy_diagnostics": df_diag.to_dict(orient="records"),
}
with open(OUT_DIR / "task2_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)

print("=== TABLE 6 PREVIEW (Mean(E/M/H) & Scenarios — Total Reward) ===")
piv = df_summary.pivot(index="scenario", columns="model", values="total_reward_mean").loc[SC_ORDER, MODEL_ORDER]
display(piv.round(4))

print("\n=== POLICY DIAGNOSTICS (ALL scenario: Entropy & Dominant Action %) ===")
display(df_diag[df_diag["scenario"] == "ALL"].sort_values(["model", "seed"]).round(4))

print("\n=== PAIRED 5-SEED TESTS: Standard A2C minus A2C_mod (Mean(E/M/H)) ===")
display(df_tests[df_tests["scenario"] == "Mean(E/M/H)"][["metric", "std_a2c_mean", "a2c_mod_mean", "delta_std_minus_mod", "ttest_p", "wilcoxon_p", "hedges_g"]].round(4))
print("OK: All Task 2 CSV & JSON outputs saved to", OUT_DIR)


=== TABLE 6 PREVIEW (Mean(E/M/H) & Scenarios — Total Reward) ===


model,Base-Stock,DQN,A2C_mod,Standard A2C
scenario,,,,
Mean(E/M/H),0.8349,0.8273,0.2556,0.8291
ALL,0.7607,0.8107,0.1717,0.8051
EASY,0.8924,0.9384,0.4123,0.9341
MEDIUM,0.8442,0.8489,0.2531,0.8528
HARD,0.7680,0.6947,0.1016,0.7005



=== POLICY DIAGNOSTICS (ALL scenario: Entropy & Dominant Action %) ===


,model,seed,scenario,mean_entropy,mean_max_prob,dom_action_idx,dom_action_pct,act13_pct
0,A2C_mod,42,ALL,0.0030,0.9988,0,81.2383,0.0000
2,A2C_mod,123,ALL,0.0003,0.9999,13,85.5700,85.5700
4,A2C_mod,256,ALL,0.0001,1.0000,13,98.6815,98.6815
6,A2C_mod,512,ALL,0.0000,1.0000,9,99.8413,0.0000
8,A2C_mod,1024,ALL,0.0068,0.9971,5,77.9870,22.0130
1,Standard A2C,42,ALL,0.0406,0.9839,0,71.3222,0.0225
3,Standard A2C,123,ALL,0.0243,0.9883,2,78.1295,0.0000
5,Standard A2C,256,ALL,0.1162,0.9487,0,65.3734,0.0207
7,Standard A2C,512,ALL,0.0278,0.9887,8,78.8059,0.0000
9,Standard A2C,1024,ALL,0.0193,0.9926,5,75.5420,0.0171



=== PAIRED 5-SEED TESTS: Standard A2C minus A2C_mod (Mean(E/M/H)) ===


,metric,std_a2c_mean,a2c_mod_mean,delta_std_minus_mod,ttest_p,wilcoxon_p,hedges_g
0,total_reward,0.8291,0.2556,0.5735,0.1385,0.0625,0.6956
1,service_level,1.0000,0.9897,0.0103,0.2910,0.2500,0.4578
2,holding_cost,0.0101,0.3834,-0.3732,0.1504,0.1250,-0.6688
3,waste_cost,0.0669,0.0622,0.0047,0.1261,0.1875,0.7260
4,ordering_cost,0.0939,0.2885,-0.1946,0.2632,0.1250,-0.4899
5,stockout_rate,0.0000,0.0103,-0.0103,0.2910,0.2500,-0.4578


OK: All Task 2 CSV & JSON outputs saved to C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task2\output
